# Tara N1 Pretraining

- **Data set:** 500M token subset of FineWeb
- **model.py:** `https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/model.py`
- **data.py:** `https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/data.py`
- **train_utils.py:** `https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/train_utils.py`
- **tara_n1_pretrain_v2.pth:** `https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/Models/tara_n1_pretrain_v2.pth`
## Completed training phases:
- Completed Shard 00 - resulting 500M token corpus
- Completed Shard 01 - resulting 1B token corpus
- Completed Shard 02 - resulting 1.5B token corpus
- Completed Shard 03 - resulting 2B token corpus
- Completed Shard 04 - resulting 2.5B token corpus
- Completed Shard 05 - resulting 3B token corpus
- Completed Shard 06 - resulting 3.5B token corpus
- Completed Shard 07 - resulting 4B token corpus
- Completed Shard 08 - resulting 4.55B token corpus
- Completed Shard 09 - resulting 5B token corpus

In [1]:
import torch
from torch import nn
import tiktoken
import requests
import os

tokenizer = tiktoken.get_encoding("gpt2")
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [2]:
model_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/model.py")
with open("model.py", "w") as f:
    f.write(model_res.text)
print("Downloaded model.py successfully.")


model_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/data.py")
with open("_data.py", "w") as f:
    f.write(model_res.text)
print("Downloaded data.py successfully.")


train_utils_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/train_utils.py")
with open("train_utils.py", "w") as f:
    f.write(train_utils_res.text)
print("Downloaded train_utils.py successfully.")

# model_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/Models/tara_n1_pretrain.pth")
# with open("tara_n1_pretrain.pth", "wb") as f:
#     f.write(model_res.content)
# print("Downloaded wieghts")

Downloaded model.py successfully.
Downloaded data.py successfully.
Downloaded train_utils.py successfully.


In [3]:
from model import *
from train_utils import *
from _data import *

In [4]:
config = GPTConfig(
    vocab_size=tokenizer.n_vocab,
    block_size=1024,
    batch_size=8,
    d_model=256,
    hidden_layers=1024,
    n_heads=4,
    n_layers=6,
)

# The Dataset

In [5]:
from datasets import load_dataset
from tqdm.auto import tqdm
import numpy as np
from kaggle_secrets import UserSecretsClient
from huggingface_hub import hf_hub_download, login

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
login(token=HF_TOKEN)

repo_id = "ShanmukhVashtav/Fineweb-edu-5B-gpt-2-tokenized"
shard_id = 0

shard_path = hf_hub_download(
    repo_id = repo_id,
    filename = f"train/shard_{shard_id:02d}.npy",
    repo_type = "dataset"
)


print(f"Downloaded Shard {shard_id:02d} at {shard_path}")

# tokens = np.load(shard_path, mmap_mode="r")



# print(f"Collected {len(tokens)} tokens.")


train/shard_00.npy:   0%|          | 0.00/1.00G [00:00<?, ?B/s]

Downloaded Shard 00 at /root/.cache/huggingface/hub/datasets--ShanmukhVashtav--Fineweb-edu-5B-gpt-2-tokenized/snapshots/462770127a52a37ac70de18092ce7fb0e6eda35c/train/shard_00.npy


In [6]:
train_split = 0.9
train_dataloader, test_dataloader = create_shards_dataloaders(shard_path, train_split, block_size = config.block_size, batch_size=config.batch_size)

Total samples: 499998976
Train samples: 449999078
Test samples: 49999898


# Pretraining the model

In [7]:
# modelV1 = CustomGPT(config)

# if torch.cuda.device_count() > 1:
#     modelV1 = nn.DataParallel(modelV1)
#     print(f"Using {torch.cuda.device_count()} GPUs")

# modelV1.to(device)

# calc_params(modelV1)

# loss_fn = nn.CrossEntropyLoss()
# optimizer = torch.optim.AdamW(modelV1.parameters(), lr=1e-4)
# scaler = torch.amp.GradScaler()


modelV2 = CustomGPT(config)
# state_dict = torch.load("tara_n1_pretrain.pth", map_location=device)
# state_dict = {k.removeprefix("module."):v for k, v in state_dict.items()}

# modelV2.load_weights("tara_n1_pretrain.pth")

if torch.cuda.device_count() > 1:
    modelV2 = nn.DataParallel(modelV2)
    print(f"Using {torch.cuda.device_count()} GPUs")

modelV2.to(device)

calc_params(modelV2)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(modelV2.parameters(), lr=1e-4)
scaler = torch.amp.GradScaler()

Using 2 GPUs
Total Parameters: 30,783,057
Trainable Parameters: 30,783,057


In [8]:
from tqdm.auto import tqdm
steps = 61035
train_iter = iter(train_dataloader)
for step in tqdm(range(1, steps+1)):
    # modelV1.train()
    modelV2.train()
    # def train_step(model, train_dataloader, train_iter, loss_fn, optimizer, scaler, device):
    # train_loss, train_iter = train_step(modelV1, train_dataloader, train_iter, loss_fn, optimizer, scaler, device)
    if not train_iter:
        train_iter = iter(train_dataloader)
    train_loss = train_step(modelV2, train_iter, loss_fn, optimizer, scaler, device, accumulation_steps=1)
    
    if step % 5000 == 0:
        # modelV1.eval()
        modelV2.eval()
        # def test_step(model, test_dl, n_steps, loss_fn, device):
        # test_loss = test_step(modelV1, test_dataloader, 20, loss_fn, device)
        test_loss = test_step(modelV2, test_dataloader, 20, loss_fn, device)
        print(f"Step {step} | Train Loss: {train_loss} | Test Loss: {test_loss:}")

  0%|          | 0/61035 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

Step 5000 | Train Loss: 6.210978984832764 | Test Loss: 6.2845723867416385


  0%|          | 0/20 [00:00<?, ?it/s]

Step 10000 | Train Loss: 5.8688249588012695 | Test Loss: 5.91619086265564


...

Step 50000 | Train Loss: 4.923012733459473 | Test Loss: 4.8311763048172


  0%|          | 0/20 [00:00<?, ?it/s]

Step 55000 | Train Loss: 4.604743003845215 | Test Loss: 4.730947208404541


  0%|          | 0/20 [00:00<?, ?it/s]

Step 60000 | Train Loss: 4.730410575866699 | Test Loss: 4.668392133712769


In [9]:
torch.save(modelV2.state_dict(), "tara_n1_pretrain_v2_00.pth")
torch.save(optimizer.state_dict(), "tara_optim_v1.pth")

# Testing



In [10]:
query = "Once upon a time, "

context = torch.tensor(tokenizer.encode(query), dtype=torch.long).unsqueeze(0).to(device)

# modelV1.eval()
test_model = CustomGPT(config)
test_model.load_weights("tara_n1_pretrain_v2_00.pth")
# test_model.load_weights("tara_n1_pretrain.pth")
test_model.to(device)
test_model.eval()
with torch.inference_mode():
    # output = modelV1.generate(context, max_new_tokens=100)
    output = test_model.generate(context)

print(f"Input:\n{query}\n")
print(f"Output:\n{tokenizer.decode(output[0].tolist())}")


Loaded weights from tara_n1_pretrain_v2_00.pth
Input:
Once upon a time, 

Output:
Once upon a time, 
This is the vocabulary test that can not be done. A middle-classule clarinet or Thro spawning-neccatic pychild2D - 6B
|Some of us may remain missing.A.S. Soc 2004 Kirmmetric J/CHOR, in London a Minneapolis Veteran crew supported a bomber from Moffuhlam at Francescageka south of Scotland, with 94 of the crew in Glazingchailik and handedo FERN in Duluthne. Besieveter desolation: A glorious division given for similar Parmariagerbnel region in Scotland, Asia Minor 33, is currently on the FIS, the OR P quadrub Chaffi) by Karin aliphensis. One of these subspecies Frontier ZEH C (Allah Abdelota Island) in Scotland in the region---Plarkin de Jerusalem in Balukm. We have arrived at one of 60 divisions, whose 2 extended,000 Mamblat has caught 96 levels. The following measures, the water remained 4 parts operating in length. In December, 12th, the Karzai Sea, the Ospinal Boku emerges north of Ely 